# 01_model

- Imports DeclipNet from util.py
- Verifies forward pass input/output shapes
- Counts trainable parameters
- Checks encoder skip connection shapes
- Confirms output range is unconstrained (no output activation)

In [1]:
import sys
import torch

sys.path.insert(0, "..")
from config import *
from util import DeclipNet

In [2]:
# Instantiate model
model = DeclipNet()

# Parameter count
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Forward pass shape check
x = torch.randn(4, 1, 1024)
with torch.no_grad():
    out = model(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, f"Shape mismatch: {out.shape} != {x.shape}"
print("Shape check passed.")

# Encoder skip connection shapes
print("\nEncoder skip shapes:")
hooks = []
skip_shapes = []

def hook_fn(module, input, output):
    skip_shapes.append(output.shape)

for encoder in model.encoders:
    hooks.append(encoder.register_forward_hook(hook_fn))

with torch.no_grad():
    _ = model(x)

for hook in hooks:
    hook.remove()

for i, shape in enumerate(skip_shapes):
    print(f"  Encoder {i+1}: {shape}")

# Output range check — should not be bounded
torch.manual_seed(0)
x = torch.randn(32, 1, 1024)
with torch.no_grad():
    out = model(x)

print(f"Output min: {out.min().item():.4f}")
print(f"Output max: {out.max().item():.4f}")
assert out.min().item() < 0, "Output has no negative values — possible unwanted activation"
assert out.max().item() > 0, "Output has no positive values — possible unwanted activation"
print("Output range check passed.")

Trainable parameters: 314,001
Input shape:  torch.Size([4, 1, 1024])
Output shape: torch.Size([4, 1, 1024])
Shape check passed.

Encoder skip shapes:
  Encoder 1: torch.Size([4, 8, 512])
  Encoder 2: torch.Size([4, 16, 256])
  Encoder 3: torch.Size([4, 32, 128])
  Encoder 4: torch.Size([4, 64, 64])
Output min: -33.0755
Output max: 27.0844
Output range check passed.


In [3]:
# Probe intermediate decoder outputs
torch.manual_seed(0)
x = torch.randn(32, 1, 1024)

activation = {}

def make_hook(name):
    def hook_fn(module, input, output):
        activation[name] = output
    return hook_fn

hooks = []
for i, decoder in enumerate(model.decoders):
    hooks.append(decoder.register_forward_hook(make_hook(f"decoder_{i+1}")))

with torch.no_grad():
    out = model(x)

for hook in hooks:
    hook.remove()

for name, act in activation.items():
    print(f"{name}: min={act.min():.4f}, max={act.max():.4f}, std={act.std():.4f}")

decoder_1: min=0.0000, max=14.8915, std=2.2505
decoder_2: min=0.0000, max=13.5030, std=1.7318
decoder_3: min=0.0000, max=15.0318, std=1.6763
decoder_4: min=-33.0755, max=27.0844, std=5.7328
